# Micro-Leo: fine-tuning di un modello piccolo per azioni di scrittura (IT/EN)

Studente (default `qwen05`; prova anche `qwen15`, `gemma1b`): l'esperimento con **`openbmb/MiniCPM5-1B`** (≈1,08 miliardi di parametri, `LlamaForCausalLM`, Apache 2.0) ha dato italiano scadente: il modello non conosce bene la lingua. Azioni: *riepilogo, spiegazione, parafrasi, migliora, cambia tono, cambia lunghezza, citazione, social media*.

> ⚠️ **Lingue:** la scheda del modello dichiara supporto per **inglese e cinese**, non per l'italiano. Il fine-tuning su dati italiani aiuta, ma la qualità in italiano va verificata con la cella di valutazione. Se è insufficiente, cambia `PRESET` in `qwen05` (Qwen2.5-0.5B, più leggero e con italiano migliore) o `gemma270` (270M).

**Differenze rispetto al notebook con Unsloth:** qui si usa solo `transformers` + `peft` (nessun Unsloth), perché il training è indipendente dal template di chat:
- il prompt viene reso dal `chat_template` del tokenizer (con `enable_thinking=False`, se il template lo supporta) e la loss è calcolata **solo sulla risposta** (etichette `-100` sul prompt), così training e inferenza usano lo stesso formato;
- l'export è GGUF via `llama.cpp` (`convert_hf_to_gguf.py` + `llama-quantize`).

**Dataset:** puoi usare `leo_dataset.jsonl` già pronto (≈560 esempi, IT/EN, scritti a mano) caricandolo su Colab: in quel caso il teacher non serve e si risparmia gran parte del tempo. Altrimenti il notebook lo genera.

**Pipeline (senza dataset pronto):** teacher (Qwen2.5-3B/7B) genera gli output di riferimento da testi sorgente (Wikipedia IT/EN + testi sintetici) → filtri → LoRA sullo studente → valutazione automatica → GGUF.

> Runtime: **GPU T4**. La generazione dati è la parte più lunga (~30-60 min; riduci `N_*` per una prova). I prompt sono identici a quelli di `leo_local.py`.

### 1. Installazione

In [ ]:
!pip install -q -U transformers peft accelerate bitsandbytes datasets torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 125.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 50.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 34.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 58.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 39.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 105.6 MB/s eta 0:00:00


### 2. Configurazione

In [ ]:
PRESET = "qwen15"        # "qwen05" | "qwen15" | "gemma1b" | "gemma270" | "minicpm1b"  (MiniCPM5-1B: solo EN/ZH, italiano scadente)
TEACHER = "qwen3b"       # "qwen3b" (più veloce) | "qwen7b" (output migliori, ~2x più lento)

# quanti testi sorgente (ogni testo produce ~4 esempi)
N_WIKI_IT, N_WIKI_EN = 250, 120
N_SYNTH_IT, N_SYNTH_EN = 220, 110
ACTIONS_PER_TEXT = 4
N_EVAL = 120             # esempi tenuti fuori per la valutazione

TEACHERS = {
    "qwen3b": "unsloth/Qwen2.5-3B-Instruct-bnb-4bit",   # checkpoint già quantizzati a 4 bit (serve bitsandbytes)
    "qwen7b": "unsloth/Qwen2.5-7B-Instruct-bnb-4bit",
}
PRESETS = {
    "minicpm1b": dict(name="openbmb/MiniCPM5-1B",        lr=2e-4, r=64, epochs=5, bs=4, accum=2),
    "qwen05":    dict(name="Qwen/Qwen2.5-0.5B-Instruct", lr=2e-4, r=64, epochs=6, bs=8, accum=1),
    "qwen15":    dict(name="Qwen/Qwen2.5-1.5B-Instruct", lr=1e-4, r=32, epochs=3, bs=4, accum=2),
    "gemma1b":   dict(name="unsloth/gemma-3-1b-it",      lr=1e-4, r=64, epochs=4, bs=4, accum=2),
    "gemma270":  dict(name="unsloth/gemma-3-270m-it",    lr=1e-4, r=64, epochs=6, bs=8, accum=1),
}
P = PRESETS[PRESET]
MAX_SEQ = 1536
SEED = 3407
LEO_DATASET = "leo_dataset.jsonl"   # se il file esiste nella cartella di Colab, teacher e generazione vengono saltati
print("Student:", P["name"], "| Teacher:", TEACHERS[TEACHER])

Student: Qwen/Qwen2.5-1.5B-Instruct | Teacher: unsloth/Qwen2.5-3B-Instruct-bnb-4bit


### 3. Prompt condivisi (identici a `leo_local.py`)

In [ ]:
import json, os, random, re, gc
random.seed(SEED)

SYSTEM = ("Sei un assistente di scrittura che lavora in locale. Rispondi SOLO con il risultato richiesto, "
          "senza premesse, commenti o virgolette di contorno. Usa la stessa lingua del testo dell'utente, "
          "salvo diversa richiesta. Non inventare fatti che non sono nel testo.")

TONES = {
    "professional": "professionale e curato",
    "casual": "informale e colloquiale",
    "enthusiastic": "entusiasta e positivo",
    "informative": "informativo e neutro",
    "creative": "creativo e vivace",
}
PLATFORMS = {
    "linkedin": "un post LinkedIn (tono professionale, 3-5 frasi, massimo 3 hashtag)",
    "x": "un post per X/Twitter (massimo 280 caratteri, massimo 2 hashtag)",
    "instagram": "una didascalia Instagram (tono coinvolgente, qualche emoji, 5-8 hashtag in fondo)",
    "facebook": "un post Facebook (tono amichevole, 2-4 frasi)",
}

def build_prompt(action: str, param: str, text: str) -> str:
    if action == "summary":
        ins = "Riassumi il testo seguente in modo chiaro e conciso (un paragrafo breve oppure al massimo 5 punti elenco)."
    elif action == "explain":
        ins = "Spiega il testo seguente in parole semplici, come a chi non conosce l'argomento. Chiarisci i termini difficili."
    elif action == "paraphrase":
        ins = "Parafrasa il testo seguente con parole diverse, mantenendo lo stesso significato e la stessa lunghezza circa."
    elif action == "improve":
        ins = "Migliora il testo seguente: correggi errori e refusi e rendi lo stile più chiaro e scorrevole, senza cambiare il significato."
    elif action == "tone" and param in TONES:
        ins = f"Riscrivi il testo seguente con un tono {TONES[param]}, mantenendo il significato."
    elif action == "length" and param == "shorter":
        ins = "Riscrivi il testo seguente in forma molto più breve, tenendo solo le informazioni essenziali."
    elif action == "length" and param == "longer":
        ins = "Riscrivi il testo seguente in forma più lunga e sviluppata, aggiungendo chiarezza e fluidità ma nessun fatto nuovo."
    elif action == "quote":
        ins = "Dal testo seguente ricava una citazione breve e memorabile (una o due frasi) che ne catturi l'idea centrale."
    elif action == "social" and param in PLATFORMS:
        ins = f"Crea {PLATFORMS[param]} basato sul testo seguente."
    else:
        raise ValueError("azione non valida")
    return f"{ins}\n\nTESTO:\n\"\"\"\n{text}\n\"\"\""

### 4. Funzioni comuni: generazione a batch, rumore, filtri, controlli

In [ ]:
import torch

def chat_generate(model, tok, messages_list, max_new_tokens=320, temperature=0.4, batch_size=24, sample=True, tk=None):
    """Genera una risposta per ogni lista di messaggi. Ordina per lunghezza per efficienza."""
    tok.padding_side = "left"
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    texts = [tok.apply_chat_template(m, tokenize=False, add_generation_prompt=True, **(tk or {})) for m in messages_list]
    order = sorted(range(len(texts)), key=lambda i: len(texts[i]))
    results = [None] * len(texts)
    for s in range(0, len(order), batch_size):
        idx = order[s:s + batch_size]
        enc = tok([texts[i] for i in idx], return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
        kw = dict(do_sample=True, temperature=temperature, top_p=0.9, repetition_penalty=1.05) if sample else dict(do_sample=False)
        with torch.no_grad():
            out = model.generate(**enc, max_new_tokens=max_new_tokens, use_cache=True,
                                 pad_token_id=tok.pad_token_id, **kw)
        for j, i in enumerate(idx):
            results[i] = tok.decode(out[j][enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        print(f"\r  {min(s + batch_size, len(order))}/{len(order)}", end="")
    print()
    return results

def add_noise(text, p=0.06):
    """Introduce refusi/errori tipici per creare input realistici per l'azione 'Migliora'."""
    out = []
    for w in text.split(" "):
        if len(w) > 3 and random.random() < p:
            k = random.randrange(4)
            i = random.randrange(1, len(w) - 1)
            if k == 0:   w = w[:i] + w[i + 1] + w[i] + w[i + 2:]          # scambio
            elif k == 1: w = w[:i] + w[i + 1:]                             # lettera mancante
            elif k == 2: w = w[:i] + w[i] + w[i:]                          # lettera doppia
            else:        w = w.translate(str.maketrans("àèéìòù", "aeeiou"))  # accenti persi
        out.append(w)
    t = " ".join(out)
    return t[0].lower() + t[1:] if random.random() < 0.3 and t else t

PREAMBLE = re.compile(r"^\s*(certo|ecco|sure|here is|here's|naturalmente|va bene|ok[,!. ]|riassunto:|riepilogo:)", re.I)

def clean_output(text):
    t = text.strip().strip('"').strip("“”").strip()
    if not t or len(t) > 2500 or PREAMBLE.match(t) or "TESTO:" in t or '"""' in t:
        return None
    return t

IT_W = set("il lo la le gli di del della che per con non una un è sono come più anche nel nella dei delle ma se ci si".split())
EN_W = set("the of and to in is that for with as on are this it be from by at or an was were which not".split())
def guess_lang(t):
    ws = re.findall(r"[a-zàèéìòù']+", t.lower())
    return "it" if sum(w in IT_W for w in ws) >= sum(w in EN_W for w in ws) else "en"

def check(action, param, inp, out):
    """Controlli automatici a regole; ritorna dict di booleani."""
    r = {"non_vuoto": bool(out.strip()), "senza_preambolo": not PREAMBLE.match(out),
         "stessa_lingua": len(out) < 60 or guess_lang(out) == guess_lang(inp), "senza_leak": "TESTO:" not in out}
    n_in, n_out = len(inp), len(out)
    if action == "summary":                 r["più_corto"] = n_out < n_in
    if action == "length" and param == "shorter": r["più_corto"] = n_out < 0.8 * n_in
    if action == "length" and param == "longer":  r["più_lungo"] = n_out > 1.1 * n_in
    if action == "social" and param == "x":       r["max_280"] = n_out <= 280
    if action == "quote":                         r["breve"] = n_out <= 300
    return r

### 4b. Dataset pronto o generazione?

Carica `leo_dataset.jsonl` (pannello *File* di Colab) nella cartella di lavoro: ogni riga ha `sid, action, param, input, output` e le celle 5-7 (teacher, testi sorgente, generazione) non vengono eseguite. Senza il file, il dataset viene generato con il teacher.

In [ ]:
import os
HAVE_DATASET = os.path.exists(LEO_DATASET)
print("Dataset pronto trovato: teacher e generazione dati verranno saltati." if HAVE_DATASET
      else "Nessun dataset trovato: lo genero con il teacher (celle 5-7).")


Dataset pronto trovato: teacher e generazione dati verranno saltati.


### 5. Teacher: carica il modello grande

In [ ]:
if not HAVE_DATASET:
    from transformers import AutoModelForCausalLM, AutoTokenizer

    t_tok = AutoTokenizer.from_pretrained(TEACHERS[TEACHER])
    t_model = AutoModelForCausalLM.from_pretrained(TEACHERS[TEACHER], device_map="cuda", torch_dtype=torch.float16).eval()
    print("Teacher pronto.")

### 6. Testi sorgente: Wikipedia (IT/EN) + testi sintetici di vari generi

In [ ]:
if not HAVE_DATASET:
    from datasets import load_dataset

    def wiki_paragraphs(lang, n):
        """Un paragrafo (250-1100 caratteri) da n articoli casuali. Streaming: niente download completo."""
        try:
            ds = load_dataset("wikimedia/wikipedia", f"20231101.{lang}", split="train", streaming=True)
            ds = ds.shuffle(seed=SEED, buffer_size=5000)
            out = []
            for ex in ds:
                paras = [p.strip() for p in ex["text"].split("\n\n") if 250 <= len(p.strip()) <= 1100 and not p.startswith("==")]
                if paras: out.append(random.choice(paras))
                if len(out) >= n: break
            return out
        except Exception as e:
            print(f"[wikipedia {lang}] non disponibile ({e}); uso solo testi sintetici")
            return []

    GENRES = {
     "it": ["email di lavoro", "recensione di un prodotto", "articolo di cronaca", "post di blog", "messaggio informale a un amico",
            "verbale di riunione", "descrizione tecnica", "paragrafo di un saggio", "annuncio immobiliare", "nota personale",
            "comunicato stampa", "risposta a un cliente"],
     "en": ["work email", "product review", "news article", "blog post", "casual message to a friend",
            "meeting minutes", "technical description", "essay paragraph", "press release", "customer support reply"],
    }
    TOPICS = ["viaggi", "cucina", "tecnologia", "salute", "sport", "finanza personale", "scuola e università", "ambiente", "musica",
              "cinema", "lavoro da remoto", "auto elettriche", "intelligenza artificiale", "storia", "salute mentale", "giardinaggio",
              "videogiochi", "moda", "sicurezza informatica", "politica locale", "startup", "fotografia", "animali domestici",
              "libri", "energia rinnovabile", "trasporti pubblici", "turismo", "matematica", "medicina", "arte contemporanea"]

    def synthetic_texts(lang, n):
        L = "italiano" if lang == "it" else "inglese"
        reqs = []
        for _ in range(n):
            g, t, w = random.choice(GENRES[lang]), random.choice(TOPICS), random.choice([60, 90, 130, 180, 240])
            reqs.append([{"role": "system", "content": "Scrivi solo il testo richiesto, senza titoli, commenti o virgolette."},
                         {"role": "user", "content": f"Scrivi in {L} un testo di circa {w} parole: {g} sul tema «{t}». Rendilo realistico e specifico."}])
        outs = chat_generate(t_model, t_tok, reqs, max_new_tokens=380, temperature=0.9)
        return [o for o in outs if 200 <= len(o) <= 1500 and guess_lang(o) == lang]

    sources = []   # (id, lang, kind, text)
    for lang, n in (("it", N_WIKI_IT), ("en", N_WIKI_EN)):
        sources += [(len(sources) + i, lang, "wiki", t) for i, t in enumerate(wiki_paragraphs(lang, n))]
    print("wiki:", len(sources))
    for lang, n in (("it", N_SYNTH_IT), ("en", N_SYNTH_EN)):
        print("sintetici", lang)
        base = len(sources)
        sources += [(base + i, lang, "synth", t) for i, t in enumerate(synthetic_texts(lang, n))]
    print("Testi sorgente totali:", len(sources))

### 7. Genera gli esempi con il teacher (azioni × testi) e filtra

In [ ]:
if not HAVE_DATASET:
    GROUPS = ["summary", "explain", "paraphrase", "improve", "tone", "length", "quote", "social"]
    MAXTOK = {"summary": 220, "explain": 320, "paraphrase": 350, "improve": 380, "tone": 380, "length": 420, "quote": 90, "social": 200}

    def sample_actions(text, k):
        n = len(text)
        ok = [g for g in GROUPS if not (g == "summary" and n < 300) and not (g == "quote" and n < 200)]
        out = []
        for g in random.sample(ok, min(k, len(ok))):
            if g == "tone":     out.append((g, random.choice(list(TONES))))
            elif g == "social": out.append((g, random.choice(list(PLATFORMS))))
            elif g == "length":
                opts = [p for p, c in (("shorter", n > 250), ("longer", n < 700)) if c]
                out.append((g, random.choice(opts or ["shorter"])))
            else:               out.append((g, ""))
        return out

    jobs = []   # (source_id, action, param, input_text)
    for sid, lang, kind, text in sources:
        for a, p in sample_actions(text, ACTIONS_PER_TEXT):
            inp = add_noise(text) if (a == "improve" and random.random() < 0.8) else text
            jobs.append((sid, a, p, inp))
    print("Richieste al teacher:", len(jobs))

    msgs = [[{"role": "system", "content": SYSTEM}, {"role": "user", "content": build_prompt(a, p, inp)}] for _, a, p, inp in jobs]
    outs = []
    for a in GROUPS:   # raggruppa per azione per usare max_new_tokens adeguato
        idx = [i for i, j in enumerate(jobs) if j[1] == a]
        if not idx: continue
        print(f"{a}: {len(idx)}")
        res = chat_generate(t_model, t_tok, [msgs[i] for i in idx], max_new_tokens=MAXTOK[a])
        outs += list(zip(idx, res))
    raw = dict(outs)

    examples, dropped = [], 0
    for i, (sid, a, p, inp) in enumerate(jobs):
        o = clean_output(raw[i])
        if o is None or not all(check(a, p, inp, o).values()):
            dropped += 1; continue
        examples.append({"sid": sid, "action": a, "param": p, "input": inp, "output": o})
    print(f"Esempi validi: {len(examples)}  (scartati dai filtri: {dropped})")

    with open("leo_dataset.jsonl", "w") as f:
        for e in examples: f.write(json.dumps(e, ensure_ascii=False) + "\n")

    # libera la GPU per lo student
    del t_model, t_tok; gc.collect(); torch.cuda.empty_cache()

### 8. Studente: modello piccolo + LoRA (transformers + peft)

In [ ]:
import json
from peft import LoraConfig, get_peft_model

from transformers import AutoModelForCausalLM, AutoTokenizer

examples = [json.loads(l) for l in open(LEO_DATASET)]
print("Esempi caricati:", len(examples))

bf16_ok = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
# Gemma 3 è instabile in fp16 -> fp32 (è piccolo); gli altri: bf16 se disponibile, altrimenti fp16
if "gemma" in P["name"]:  dtype = torch.float32
else:                     dtype = torch.bfloat16 if bf16_ok else torch.float16

tokenizer = AutoTokenizer.from_pretrained(P["name"])
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(P["name"], torch_dtype=dtype, device_map="cuda")
print("Architettura:", model.config.architectures, "| parametri: %.0fM" % (sum(p.numel() for p in model.parameters()) / 1e6))

model.gradient_checkpointing_enable()
model.enable_input_require_grads()
model = get_peft_model(model, LoraConfig(r=P["r"], lora_alpha=P["r"], lora_dropout=0.0, bias="none",
                                         target_modules="all-linear", task_type="CAUSAL_LM"))
for p in model.parameters():          # adapter in fp32: necessario con fp16 + GradScaler
    if p.requires_grad: p.data = p.data.float()
model.print_trainable_parameters()

# Il template può supportare la modalità "thinking": la disattiviamo, coerentemente in training e inferenza
TK = {"enable_thinking": False} if "enable_thinking" in str(tokenizer.chat_template) else {}
print("Argomenti template:", TK)

Esempi caricati: 563


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Architettura: ['Qwen2ForCausalLM'] | parametri: 1544M
trainable params: 36,929,536 || all params: 1,580,643,840 || trainable%: 2.3364
Argomenti template: {}


### 9. Dataset (loss solo sulla risposta) e training

In [ ]:
from datasets import Dataset
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq

def student_msgs(a, p, inp):
    return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": build_prompt(a, p, inp)}]

def end_suffix(tok):
    """Testo che il template mette dopo la risposta dell'assistant (es. il token di fine turno)."""
    m = student_msgs("summary", "", "x") + [{"role": "assistant", "content": "AAA"}]
    full = tok.apply_chat_template(m, tokenize=False, **TK)
    s = full[full.index("AAA") + 3:].rstrip()
    return s or tok.eos_token
END = end_suffix(tokenizer)
print("Suffisso di fine risposta:", repr(END))

def encode(e):
    prompt = tokenizer.apply_chat_template(student_msgs(e["action"], e["param"], e["input"]),
                                           tokenize=False, add_generation_prompt=True, **TK)
    p_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
    c_ids = tokenizer(e["output"] + END, add_special_tokens=False)["input_ids"]
    ids = (p_ids + c_ids)[:MAX_SEQ]
    labels = ([-100] * len(p_ids) + c_ids)[:MAX_SEQ]      # niente loss sul prompt
    return {"input_ids": ids, "labels": labels, "attention_mask": [1] * len(ids)}

# split per testo sorgente: nessuna variante dello stesso testo sia in train che in eval
sids = sorted({e["sid"] for e in examples}); random.shuffle(sids)
eval_sids, n_ev = set(), 0
for s in sids:
    if n_ev >= N_EVAL: break
    eval_sids.add(s); n_ev += sum(e["sid"] == s for e in examples)
train_ex = [e for e in examples if e["sid"] not in eval_sids]
eval_ex  = [e for e in examples if e["sid"] in eval_sids]
print(f"train={len(train_ex)}  eval={len(eval_ex)}")

train_ds = Dataset.from_list([encode(e) for e in train_ex])
val_ds   = Dataset.from_list([encode(e) for e in eval_ex[:60]])

# controllo visivo: prompt (mascherato) e risposta (addestrata)
ex = train_ds[0]
print("PROMPT  :", tokenizer.decode([t for t, l in zip(ex["input_ids"], ex["labels"]) if l == -100])[-350:])
print("RISPOSTA:", tokenizer.decode([l for l in ex["labels"] if l != -100])[:350])

# Calculate warmup steps
num_train_steps = (len(train_ds) // (P["bs"] * P["accum"])) * P["epochs"]
warmup_steps = int(num_train_steps * 0.05)

trainer = Trainer(
    model=model, train_dataset=train_ds, eval_dataset=val_ds,
    data_collator=DataCollatorForSeq2Seq(tokenizer, padding=True, label_pad_token_id=-100),
    args=TrainingArguments(
        output_dir="outputs", per_device_train_batch_size=P["bs"], gradient_accumulation_steps=P["accum"],
        num_train_epochs=P["epochs"], learning_rate=P["lr"], lr_scheduler_type="cosine", warmup_steps=warmup_steps,
        weight_decay=0.01, optim="adamw_torch",
        bf16=(dtype == torch.bfloat16), fp16=(dtype == torch.float16),
        logging_steps=10, eval_strategy="epoch", save_strategy="epoch", save_total_limit=2,
        load_best_model_at_end=True, metric_for_best_model="eval_loss", greater_is_better=False,   # tiene l'epoca migliore
        report_to="none", seed=SEED,
    ),
)
# Baseline zero-shot PRIMA del fine-tuning: se qui l'italiano è già pessimo, il fine-tuning non basta (cambia PRESET)
model.gradient_checkpointing_disable(); model.config.use_cache = True; model.eval()
_b = chat_generate(model, tokenizer, [student_msgs(e["action"], e["param"], e["input"]) for e in eval_ex[:4]],
                   max_new_tokens=200, batch_size=4, sample=False, tk=TK)
for e, o in zip(eval_ex[:4], _b):
    print(f"[{e['action']}{'/' + e['param'] if e['param'] else ''}] ZERO-SHOT: {o[:300]}\n")
model.train(); model.gradient_checkpointing_enable(); model.enable_input_require_grads(); model.config.use_cache = False

trainer.train()

Suffisso di fine risposta: '<|im_end|>'
train=440  eval=123
PROMPT  : ché Marco e Giulia sono in trasferta dal cliente fino a giovedì sera. L'agenda resta la stessa: stato dei lavori sul portale, budget del prossimo trimestre e assegnazione dei nuovi ticket. Vi chiedo di confermare la presenza entro domani e di caricare le vostre slide nella cartella condivisa prima dell'incontro.
"""<|im_end|>
<|im_start|>assistant

RISPOSTA: La riunione di allineamento di giovedì alle 10 è spostata a venerdì alle 15 perché Marco e Giulia sono in trasferta. L'agenda non cambia (portale, budget, ticket). Occorre confermare la presenza entro domani e caricare le slide nella cartella condivisa.<|im_end|>
  4/4
[summary] ZERO-SHOT: Luminoso bilocale di 60 mq al terzo piano con ascensore, in zona Navigli a Milano. Compreso ingresso, soggiorno con cucina, camera matrimoniale e bagno, completamente arredata con AC e balcone. Spese condominiali inclusa, disponibile da aprile. Idealità per singoli o coppie.

[

Epoch,Training Loss,Validation Loss
1,0.999767,0.947041
2,0.715468,0.954303
3,0.494718,0.988101


TrainOutput(global_step=165, training_loss=0.7361951380064993, metrics={'train_runtime': 2327.137, 'train_samples_per_second': 0.567, 'train_steps_per_second': 0.071, 'total_flos': 3562710390632448.0, 'train_loss': 0.7361951380064993, 'epoch': 3.0})

### 10. Valutazione automatica sul set tenuto fuori

In [ ]:
from collections import defaultdict
model.gradient_checkpointing_disable(); model.config.use_cache = True; model.eval()

preds = chat_generate(model, tokenizer, [student_msgs(e["action"], e["param"], e["input"]) for e in eval_ex],
                      max_new_tokens=400, batch_size=16, sample=False, tk=TK)

score = defaultdict(lambda: [0, 0])
for e, o in zip(eval_ex, preds):
    for k, v in check(e["action"], e["param"], e["input"], o).items():
        score[k][0] += bool(v); score[k][1] += 1
print("Controlli automatici (student, greedy):")
for k, (ok, n) in sorted(score.items()):
    print(f"  {k:16s} {ok}/{n}  {ok/n:.0%}")

# --- diagnosi: l'output è una copia dell'input? (per le azioni che dovrebbero trasformare il testo)
import difflib, re
sim = defaultdict(list)
for e, o in zip(eval_ex, preds):
    if e["action"] in ("tone", "paraphrase", "social", "explain", "quote", "summary"):
        sim[e["action"] + ("/" + e["param"] if e["param"] else "")].append(difflib.SequenceMatcher(None, e["input"], o).ratio())
print("\nSimilarità output-input (1.0 = copia identica; nel dataset del teacher l'ideale è molto più basso):")
for a, v in sorted(sim.items()):
    print(f"  {a:22s} media {sum(v)/len(v):.2f} | quasi-copie (>0.85): {sum(x > 0.85 for x in v)}/{len(v)}")
ig = [o for e, o in zip(eval_ex, preds) if e["action"] == "social" and e["param"] == "instagram"]
if ig: print("Instagram con 5-8 hashtag:", sum(5 <= len(re.findall(r"#\w+", o)) <= 8 for o in ig), "/", len(ig))


  123/123
Controlli automatici (student, greedy):
  breve            11/11  100%
  max_280          6/6  100%
  non_vuoto        123/123  100%
  più_corto        24/24  100%
  più_lungo        7/8  88%
  senza_leak       123/123  100%
  senza_preambolo  123/123  100%
  stessa_lingua    121/123  98%

Similarità output-input (1.0 = copia identica; nel dataset del teacher l'ideale è molto più basso):
  explain                media 0.11 | quasi-copie (>0.85): 0/7
  paraphrase             media 0.13 | quasi-copie (>0.85): 0/7
  quote                  media 0.37 | quasi-copie (>0.85): 0/11
  social/facebook        media 0.32 | quasi-copie (>0.85): 0/5
  social/instagram       media 0.07 | quasi-copie (>0.85): 0/6
  social/linkedin        media 0.42 | quasi-copie (>0.85): 0/3
  social/x               media 0.47 | quasi-copie (>0.85): 0/6
  summary                media 0.26 | quasi-copie (>0.85): 0/15
  tone/casual            media 0.33 | quasi-copie (>0.85): 0/7
  tone/creative          media

### 11. Esempi a confronto (input → teacher → student)

In [ ]:
for e, o in random.sample(list(zip(eval_ex, preds)), 6):
    print("=" * 90)
    print(f"[{e['action']}{'/' + e['param'] if e['param'] else ''}]  INPUT: {e['input'][:300]}")
    print("--- TEACHER:", e["output"][:400])
    print("--- STUDENT:", o[:400])

[tone/casual]  INPUT: Per reimpostare la password vai nella pagina di accesso e clicca su «Password dimenticata». Inserisci l'indirizzo email con cui ti sei registrato e riceverai un messaggio con un link valido per un'ora. Apri il link, scegli una nuova password di almeno dodici caratteri e conferma. Se non trovi l'emai
--- TEACHER: Hai dimenticato la password? Nessun problema: nella pagina di accesso premi «Password dimenticata», scrivi la tua email e ti arriva un link che funziona per un'ora. Aprilo e metti una password nuova, lunga almeno dodici caratteri. Se l'email non si vede, guarda nello spam!
--- STUDENT: Vai alla tua pagina di accesso e clicca sul pulsante "Password dimenticata". Scriviamo l'indirizzo email usato per registrarci e ci arriva un messaggio con un link valido per un'ora. Apri il link, scegliamo una nuova password di almeno dodici caratteri e confermiamola. Se non troviamo l'email, controlliamo la cartella dello spam.
[social/facebook]  INPUT: Da qualche mese ho 

### 12. Export: merge LoRA → GGUF con llama.cpp

In [ ]:
model.save_pretrained("micro_leo_lora"); tokenizer.save_pretrained("micro_leo_lora")   # adapter (leggero)

merged = model.merge_and_unload()
merged.save_pretrained("micro_leo_merged", safe_serialization=True)
tokenizer.save_pretrained("micro_leo_merged")
print("Modello unito salvato in micro_leo_merged")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Modello unito salvato in micro_leo_merged


In [ ]:
!pip install -q gguf sentencepiece
!git clone -q --depth 1 https://github.com/ggml-org/llama.cpp
!mkdir -p micro_leo_gguf
!python llama.cpp/convert_hf_to_gguf.py micro_leo_merged --outtype f16 --outfile micro_leo_gguf/micro-leo-f16.gguf
!cmake -S llama.cpp -B llama.cpp/build -DGGML_CUDA=OFF -DLLAMA_CURL=OFF > /dev/null
!cmake --build llama.cpp/build --target llama-quantize -j 4 > /dev/null
!llama.cpp/build/bin/llama-quantize micro_leo_gguf/micro-leo-f16.gguf micro_leo_gguf/micro-leo-q8_0.gguf q8_0
!llama.cpp/build/bin/llama-quantize micro_leo_gguf/micro-leo-f16.gguf micro_leo_gguf/micro-leo-q4_k_m.gguf q4_k_m
!ls -lh micro_leo_gguf

**Uso in locale.** Il template di chat è incorporato nel GGUF: usa `llama-server` con `--jinja` e disattiva il thinking se il modello lo prevede (vedi `Argomenti template` sopra):

```bash
llama-server -m micro-leo-q8_0.gguf --jinja --port 8081 -c 2048
CHAT_KWARGS='{"enable_thinking": false}' BASE_URL=http://localhost:8081/v1 MODEL=local python3 leo_local.py
```

Con Ollama serve scrivere a mano un `TEMPLATE` nel Modelfile e non si può passare `enable_thinking`: per questo modello è consigliato `llama-server`. `q8_0` è la scelta più fedele; `q4_k_m` è circa metà dello spazio con una perdita di qualità da verificare.

In [ ]:
# Facoltativo: salva su Google Drive (dataset, adapter e GGUF)
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r leo_dataset.jsonl micro_leo_lora micro_leo_gguf /content/drive/MyDrive/